# FSD Cube: induced current, amplifier behavior and digitized readout

There are two distinct modes:

1. **Illustrative, CPU-only:** analytic collection-like and bipolar induction-like pulses, a schematic muon, and a noise-free threshold/hold/reset model. This explains behavior; it is **not a Geant4/larnd-sim prediction**. Geometry and selected ADC transfer settings come from YAML. Pulse timing/amplitudes are chosen for illustration and are not calculated from the displayed track.
2. **Real larnd-sim capture:** load current arrays at the electronics input and the actual ADC results from the opt-in GPU capture wrapper. Read `README.md` for its command. A capture does not contain the internal amplifier state or individual random reset times. Its middle panel is explicitly an **unreset integral**, not a reconstructed CSA waveform.

The three right-hand time axes are linked: zooming one zooms all. Select a pixel using the controls; its 3D marker is highlighted.

In [ ]:
from pathlib import Path
import sys, os
# Works when Jupyter starts at the repository root or visualizations/.
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'visualizations/geometry.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter from dune-nd-mip-calib or visualizations/')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
LARND_REPO = Path(os.environ.get('LARND_SIM_REPO', str(ROOT.parent / 'larnd-sim'))).expanduser()
OUT = ROOT / 'visualizations/outputs'
OUT.mkdir(exist_ok=True)
import plotly.io as pio
pio.renderers.default = 'plotly_mimetype'
from IPython.display import display
from visualizations.geometry import load_geometry, geometry_figure, pixel_figure, tile_pixels, repositioning_boxes, export_html
print('Reading larnd-sim configuration from', LARND_REPO)

from visualizations.signals import demo_bundle, signal_figure, load_bundle, save_bundle, validate_bundle, attach_event_segments, read_channel_packets, packet_figure
fsd_cube = load_geometry(LARND_REPO, 'fsd_cube')

In [ ]:
demo = demo_bundle(fsd_cube)
print(validate_bundle(demo)['description'])
print(list(zip(demo['pixel_id'], demo['labels'])))
example = signal_figure(demo, channel=0)
example.show()

## Explore the teaching model

Try the induction-only-like channel with reset disabled and enabled. Its full current integral is approximately zero, but thresholding, resets and timing can affect what is digitized. The model omits noise and buffer response and does not duplicate all larnd-sim timing details. A zero reset period disables periodic resets. These controls use a 0.1 µs clock.

In [ ]:
import ipywidgets as widgets
pixel_control = widgets.Dropdown(options=[(str(label),i) for i,label in enumerate(demo['labels'])], description='Pixel:')
period_control = widgets.IntSlider(value=512,min=0,max=1024,step=1,description='Reset cycles:',continuous_update=False)
phase_control = widgets.FloatSlider(value=0,min=0,max=51.2,step=.1,description='Phase µs:',continuous_update=False)
threshold_control = widgets.FloatSlider(value=5000,min=1000,max=12000,step=500,description='Threshold e−:',continuous_update=False)
active_demo = {'bundle':demo, 'figure':example}
def draw_demo(channel,cycles,phase,threshold):
    b = demo_bundle(fsd_cube,reset_period_us=cycles*.1,reset_phase_us=phase,threshold_e=threshold)
    fig = signal_figure(b,channel)
    active_demo.update(bundle=b,figure=fig)
    fig.show()
demo_output = widgets.interactive_output(draw_demo, dict(channel=pixel_control, cycles=period_control,phase=phase_control,threshold=threshold_control))
display(widgets.VBox([widgets.HBox([pixel_control,period_control]),widgets.HBox([phase_control,threshold_control])]), demo_output)

## Open an actual capture

Set `TRACE_PATH` to a `.npz` generated by `python -m visualizations.capture_larnd`. Optionally set `EDEP_INPUT` to the same converted input HDF5 used for that simulation; this adds the captured event's original energy-deposit segments for spatial context. Those are the whole event (bounded to 20,000 segments), not a claim that every segment contributed to this pixel.

Capture times are local simulation microseconds, before event offsets and packet-clock conversion. The wrapper selects actual channels but does not classify them as collection/induction. Select explicit pixel IDs when that distinction matters. The full simulation must have far-field induction enabled if those signals are desired.

In [ ]:
TRACE_PATH = None  # e.g. Path('/path/to/traces/trace_00000_event_0.npz')
EDEP_INPUT = None  # e.g. Path('/path/to/muons.EDEPSIM.hdf5')
real_bundle = None
if TRACE_PATH is not None:
    real_bundle = load_bundle(TRACE_PATH)
    if EDEP_INPUT is not None:
        real_bundle = attach_event_segments(real_bundle, EDEP_INPUT)
    display(validate_bundle(real_bundle))
    captured_pixel = widgets.Dropdown(options=[(str(pid),i) for i,pid in enumerate(real_bundle['pixel_id'])],description='Pixel ID:')
    real_output = widgets.interactive_output(lambda channel: signal_figure(real_bundle,channel).show(),{'channel':captured_pixel})
    display(captured_pixel,real_output)
else:
    print('No real capture selected. All plots above are explicitly illustrative.')

## Inspect serialized charge packets separately

Set the output LARNDSIM file and exact electronics address. `packet_type=1` is the local FSD Cube LArPix-v3 data-packet setting; FSD LArPix-v2 uses 0. Verify your run configuration. No aliases, clock resets, rollover corrections, event offsets, or packet-to-capture time alignment are guessed. Reading is limited to 5,000 matching packets; choose a small pilot file. This plot uses raw ADC words, including the pedestal.

For a captured pixel, its nominal address is available as `real_bundle['addresses'][channel]`. Pre-serialization diagnostic ADCs may differ from saved packet counts because of channel masking or routing.

In [ ]:
PACKETS_PATH = None
ADDRESS = (1, 1, 11, 0)  # io_group, io_channel, chip_id, channel_id
DATA_PACKET_TYPE = 1
if PACKETS_PATH is not None:
    packets = read_channel_packets(PACKETS_PATH, ADDRESS, data_packet_type=DATA_PACKET_TYPE)
    print('Matched packets (capped at 5000):',len(packets))
    if len(packets): packet_figure(packets,clock_cycle_us=.1).show()

## Export

The HTML captures the currently selected teaching-example settings. `save_bundle(path, bundle)` can also save that example (or a capture with spatial context); it refuses to overwrite an existing bundle. An optional real-capture export below uses channel 0.

In [ ]:
print(export_html(active_demo['figure'], OUT/'signal_readout_demo.html'))
# save_bundle(OUT/'my_demo.npz', active_demo['bundle'])
if real_bundle is not None:
    print(export_html(signal_figure(real_bundle,captured_pixel.value),OUT/'signal_readout_capture.html'))

## Interpreting the panels

- Current is signed, in electron-charge units per microsecond. A charge-collecting pixel also senses induction before arrival. There are no separate wire-style induction planes here.
- In demo mode the middle panel shows the model's accumulated amplifier charge and its threshold/reset marks. In capture mode it shows only the numerical current integral; do not interpret it as the actual reset-sensitive amplifier state.
- ADC points are discrete readings, not samples of a continuous current waveform. Zero current does not imply zero pedestal-subtracted electronics output in a real noisy run.
- The captured source code hash, commit, command, event, channel IDs, and electronics constants travel with the NPZ. Shared input events are not independent physics samples.
- The capture adapter is tested against the local `do_digitize_and_update` interface. A changed larnd-sim branch may need an adapter update. A CUDA run is required to validate the adapter on your production environment.